# Agenda: CSV files

1. A little background on CSV
2. `read_csv` and `to_csv`
3. filespec
4. `sep`
5. `header`
6. `index_col`
7. `usecols`
8. `names`
9. `dtype`
10. `engine`
11. `parse_dates`
12. `date_format`
13. `comment`

# What are CSV files?

CSV originally stood for "comma-separated values," but now I've heard it can be called "character-separated values."

The idea is that we have a text-file format that contains our records:

- Every record is one line
- On each line, we have multiple fields
- Fields are separated by commas

On the face of it, that seems reasonable! But there are a few problems:

- Because it's a text format, not a binary format, Pandas needs to figure out what type of values should be in each column
- Sometimes, we (or the authors of the file) want to use a different separator. I'm partial to `'\t'` (tab), because it's unlikely to be used in the data.
- Sometimes, the column names are included, and sometimes they aren't.

You might think that CSV files aren't that complex; once you know the separator and a few other pieces of information, you should be able to read it into Python with `str.split`. 

The problem is that CSV allows us to have commas (or whatever separators we're using) *inside* of a data field.

`read_csv` handles the separator inside of a field just fine.

In [ ]:
s = 'ab,cde,fg'

s.split(',')

In [ ]:
# we can include , inside of a field if we use "" around it
s = 'ab,"cd,e",fg'
s.split(',')

# To read a CSV file, we use `pd.read_csv`

Notice that this is a function defined in the `pd` namespace, not a method we call on a data frame. The point is to create a data frame where there wasn't one.

In [ ]:
import pandas as pd
from pandas import Series, DataFrame

help(pd.read_csv)

# How can I use `read_csv`?

The most usual case is to call `pd.read_csv` with a string, a filename on the local filesystem that is in CSV format.  The file can actually be compressed with zip or a few other compression systems, and Pandas will know what to do.

But you can actually pass two other types of values:

- An open file object ("file-like object"), and if it's readable, then it'll work.
- A string containing a URL. `read_csv` will download the content from that URL, and then feed that into `read_csv`.

In [ ]:
df = pd.read_csv('albany,ny.csv')

In [ ]:
df.head()

# What do we check when we've loaded a CSV file?

1. Did we get the separator right?
2. Are the column names right? Did we get the right row for the column names?
3. Did Pandas guess the dtypes correctly?

In [ ]:
df.dtypes

# Writing CSV files

If I want to write a data frame in memory out to a CSV file, I can use `pd.to_csv`. This also has many options, but not quite as many.


In [ ]:
df.to_csv('output.csv')

In [ ]:
!head output.csv

# Exercise: Reading a CSV file

1. In the directory, alongside the Jupyter notebook for this class, you'll find the weather info for Albany, NY. Download it, and create a data frame with it.
2. Write out the data frame to disk, but using `'\t'` as the separator, rather than the default of `,`.

In [ ]:
filename = 'albany,ny.csv'

df = pd.read_csv(filename)

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.to_csv('albany.csv', sep='\t')

In [ ]:
!head albany.csv

In [ ]:
df.to_csv('albany.csv', sep='e')

In [ ]:
!head albany.csv

In [ ]:
!head $filename

In [ ]:
# what will happen in this case, if we try to read the CSV file?

df = pd.read_csv(filename)

In [ ]:
# we can ignore the first few lines of a file, telling Pandas that the headers
# are actually on another line, with the header= keyword argument.

# - Not passing it means that the first line of the file contains the headers
# - Passing it with a value of None means that there are no headers in the file
# - Passing it with an integer value indicates where the headers are

df = pd.read_csv(filename, header=2)  # this means: ignore the first 2 lines, and the 3rd contains the header names

In [ ]:
df.head()

In [ ]:
!head "albany,ny.csv"

# We can choose columns

We usually don't need all of the columns from a file. We can choose which should be brought into a data frame with the `usecols` keyword argument. This can take names or integers (starting with 0) to indicate which columns we want.

In [ ]:
df = pd.read_csv('nyc_taxi_2020-07.csv',
                 usecols=['passenger_count', 'trip_distance', 'total_amount'])

In [ ]:
df.head()

In [ ]:
df = pd.read_csv('nyc_taxi_2020-07.csv',
                 usecols=[10, 8, 9])

In [ ]:
df.head()

# No headers?

If you're reading a file without any headers, then you can specify the columns you want integers, but then assign names of your choosing. You can do this with the `name` keyword argument. Just pass a list (of the same length) of the columns' names.

In [ ]:
df = pd.read_csv('nyc_taxi_2020-07.csv',
                 usecols=[10, 8, 9],
                names=['first', 'second', 'third'])
df.head()

In [ ]:
!unzip airports.zip

In [ ]:
!mv airports/*.dat .

In [ ]:
!rmdir airports


In [ ]:
!rm airports.zip

In [ ]:
!head -20 airlines.dat

# Exercise: Airline data

1. Create a data frame with information from `airlines.dat`.
2. We only need four columns: The long, full name, the short (three-letter) name, if it exists, the short (two-letter) name, if it exists, and then the country in which it's based.
3. What 10 countries have the greatest number of airlines?

In [ ]:
filename = 'airlines.dat'

df = pd.read_csv(filename,
                header=None,
                usecols=[1, 3, 4, 6],
                names=['name', '2LC', '3LC', 'country'])

df.head(20)

In [ ]:
df['country'].value_counts().head(20)

In [ ]:
!ls *,*

In [ ]:
!head boston,ma.csv

In [ ]:
!head chicago,il.csv

In [ ]:
df = pd.read_csv('chicago,il.csv',
                 usecols=[0, 1, 2],
                 header=0,
                 names=['date_time', 'maxtemp', 'mintemp'])
df

In [ ]:
# if we have many files that fit the same filename pattern,
# and we want to turn them all into data frames, and then
# we want to combine those data frames into a single data frame, we can use a list comprehension

import glob

all_dfs = [pd.read_csv(one_filename,
                       header=0,
                       usecols=[0,1,2],
                       names=['date_time', 'maxtemp', 'mintemp'])
                       .assign(city=one_filename,
                              state=one_filename)
           for one_filename in glob.glob('*,*.csv')]



In [ ]:
# let's create a dict, with a dict comprehension!

all_dfs = {one_filename : pd.read_csv(one_filename,
                       header=0,
                       usecols=[0,1,2],
                       names=['date_time', 'maxtemp', 'mintemp'])
                       .assign(city=one_filename,
                              state=one_filename)
           for one_filename in glob.glob('*,*.csv')}



In [ ]:
# pd.concat creates one new data frame from all those in its list
pd.concat(all_dfs.values())

# dtype

Every column has a dtype. How is it set? By default, Pandas will examine all of the values in a column, and then decide:

- If it has only digits, then it uses int64
- If it has digits + a decimal point, then it uses float64
- In all other cases, it uses strings (`object`)

This has at least two drawbacks:

1. It takes time for Pandas to examine and decide for a column.
2. Its defaults are terrible for many values.
3. If you have a lot of data, then Pandas will load it in chunks. If you have different values in different rows, then it might get confused, and assign the wrong dtype. If this happens, you'll typically get a warning telling you that it ran out of memory. You can then specify the dtypes or pass `low_memory=False`.

We can avoid these issues by passing a `dtype` keyword argument to `read_csv`. `dtype` takes a dict; the keys are the column names and the values are the dtypes we want to associate with them.

In [ ]:
df = pd.read_csv('nyc_taxi_2020-07.csv',
                 usecols=['passenger_count', 'trip_distance', 'total_amount'],
                 dtype={'passenger_count':pd.Int64Dtype(),
                        'trip_distance':'float',
                        'total_amount':'float'})

In [ ]:
df.head()

In [ ]:
df.dtypes

# Exercise: Specifying dtypes

1. Read `nyc_taxi_2020-07.csv` into a data frame. How long does it take? (You can use `%timeit` to find out.)
2. Now read only four columns into the data frame -- `passenger_count`, `trip_distance`, `total_amount`, `tip_amount`. Does Pandas read this faster now?
3. Specify their dtypes, and time the loading. Is it even faster?

In [ ]:
%timeit pd.read_csv('nyc_taxi_2020-07.csv', low_memory=False)

In [ ]:
%%timeit

pd.read_csv('nyc_taxi_2020-07.csv', 
                    usecols=['passenger_count', 'trip_distance',
                             'total_amount', 'tip_amount'],
                    low_memory=False)

In [ ]:
%%timeit

pd.read_csv('nyc_taxi_2020-07.csv', 
                    usecols=['passenger_count', 'trip_distance',
                             'total_amount', 'tip_amount'],
                    dtype={'passenger_count':float,
                           'trip_distance': float,
                           'total_amount':float,
                           'tip_amount':float},
                    low_memory=False)

# PyArrow can help us to load faster

Right now, we're (by default) using the Pandas engine to load CSV files. But PyArrow, which is a super-fast in-memory storage system, has its own loader for CSV. We can specify that we want to use it by saying `engine='pyarrow'`.

In [ ]:
%timeit pd.read_csv('nyc_taxi_2020-07.csv', engine='pyarrow')

In [ ]:
%%timeit

pd.read_csv('nyc_taxi_2020-07.csv', 
                    usecols=['passenger_count', 'trip_distance',
                             'total_amount', 'tip_amount'],
                    engine='pyarrow')

In [ ]:
%%timeit

pd.read_csv('nyc_taxi_2020-07.csv', 
                    usecols=['passenger_count', 'trip_distance',
                             'total_amount', 'tip_amount'],
                    dtype={'passenger_count':float,
                           'trip_distance': float,
                           'total_amount':float,
                           'tip_amount':float},
                    engine='pyarrow')

# Dates and times

If we use the builtin Pandas engine for reading CSV files, it will not try to parse datetime information. PyArrow's parser will, however.

If you want to tell Pandas that a column should be interpreted as a datetime, do that by passing a list of strings (column names) to `parse_dates`.

In [ ]:
df = pd.read_csv('nyc_taxi_2020-07.csv', 
                    usecols=['tpep_pickup_datetime',
                             'tpep_dropoff_datetime',
                             'passenger_count', 'trip_distance',
                             'total_amount', 'tip_amount'],
                    parse_dates=['tpep_pickup_datetime',
                                 'tpep_dropoff_datetime'],
                    low_memory=False)
df.head()

In [ ]:
df.dtypes

# Ways to affect the parsing of dates

1. `parse_dates`, pass it a list of column names (or numbers) that should be parsed as dates
2. `dayfirst`, which defaults to `False`, indicates whether XX-YY-ZZZZ dates should be interpreted as American dates (with the month first) or European dates (with the day first).
3. `date_format`, where we can specify what the datetime format should look like.

In [ ]:
!head -20 albany,ny.csv

# Exercise: Albany temps and snowfall

1. Read the Albany, NY data into a data frame. Make sure that `date_time` is treated as a `datetime` value.
2. You can extract the month from `datetime` with the `.dt.month` method. Use that to retrieve the month, and then calculate the average snowfall in Albany for each month.

In [ ]:
filename = 'albany,ny.csv'

df = pd.read_csv(filename, header=2)
df.info(memory_usage='deep')

In [ ]:
filename = 'albany,ny.csv'

df = pd.read_csv(filename, header=2, parse_dates=['date_time'])
df.info(memory_usage='deep')

In [ ]:
df.groupby(df['date_time'].dt.month)['albany,ny_totalSnow_cm'].mean()

# Non-CSV CSV files

You can treat a file as a CSV file if it has (a) regular separators and (b) can be handled with the various options.

In [ ]:
!head -20 /etc/passwd

In [ ]:
pd.read_csv('/etc/passwd',
            sep=':',
           comment='#',
           header=None,
           names=['username', 'pw', 'uid', 'gid', 'name', 'homedir', 'shell'])